# STA/LTA detection with every trained model

| Part | Switch | Device | What it does |
|---|---|---|---|
| 0 | always | CPU | environment, repo at the pinned commit, `wdn_model_ext.py`, helpers |
| 1 | `RUN_SELFTEST` | CPU | load every model, forward shapes, parameter counts, STA/LTA matches ObsPy, stitching is exact |
| 2 | `RUN_DOWNLOAD` | CPU + internet | continuous data per day, station metadata, catalogue, travel times |
| 3 | `RUN_REAL` | GPU (CPU works) | separate the real data with every model, STA/LTA sweep, catalogue matching |
| 4 | `RUN_SYNTH` | GPU (CPU works) | long synthetic streams with known onsets, same pipeline |
| 5 | `RUN_FIGURES` | CPU | tables and figures for the report, same style and colours as notebook 07 |
| 6 | `RUN_PAPER_FIGS` | GPU (CPU works) | STA/LTA figures in the layout of the paper (step7a): ratio + 3 components, raw vs separated, and triggers per day |

In [ ]:
# SWITCHES
RUN_SELFTEST = True    # Part 1: load every model, STA/LTA and stitching checks
RUN_DOWNLOAD = True    # Part 2: continuous data, station metadata, catalogue
RUN_REAL = True        # Part 3: separate the real data with every model, STA/LTA, catalogue matching
RUN_SYNTH = True       # Part 4: synthetic long streams from the STEAD / INSTANCE test splits
RUN_FIGURES = True     # Part 5: tables and figures
RUN_PAPER_FIGS = True  # Part 6: STA/LTA figures in the paper's layout
RESUME = True

# Models
INCLUDE_PRETRAINED = True
ONLY_MODELS = None
EPOCHS_REQUIRED = 100
BATCH = 256
MODELS_DIR = None

# Real data
STATIONS = [dict(net="HV", sta="HAT", loc="*", cha="HH?"), dict(net="IU", sta="POHA", loc="00", cha="BH?"),]
START = "2021-08-02"
N_DAYS = 14
ROTATE_TO_ZNE = False
CATALOG = "repo"
CATALOG_RADIUS_DEG = 5.0
CATALOG_MIN_MAG = 0.0
KEEP_RAW_MSEED = False
FDSN_CLIENTS = ("EARTHSCOPE", "IRIS")
NORM = "window"
OVERLAP = 0.5

# STA/LTA (paper step7a)
STA_S, LTA_S = 2.0, 60.0
TRIG_ON, TRIG_OFF = 6.0, 2.0
TRIG_ON_SWEEP = (2.5, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0, 15.0)
COINCIDENCE = 1
BANDPASS_HZ = (0.5, 4.9)

# Catalogue matching (real data), seconds
MATCH_PRE, MATCH_POST = 5.0, 10.0
EVENT_TAIL = 30.0
MAG_BINS = (-10, 1.0, 2.0, 3.0, 10)
N_EXAMPLE_EVENTS = 4

# Synthetic streams
SYN_SOURCES = ("STEAD", "INSTANCE")
SYN_POOL_ROWS = 4000
SYN_EVENTS = 1400
SYN_SNR = (0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0)
SYN_GAP_S = (90.0, 240.0)
SYN_STREAM_H = 6.0
SYN_CROSSFADE_S = 5.0
SYN_ONSET_RANGE_S = (3.0, 40.0)
SYN_DETECT = (2.0, 5.0)
SYN_SPAN_PAD = (3.0, 10.0)
SYN_SEED = 2026

FIG_MODELS_HOURLY = ("pretrained_LSTM", "stead_LSTM_seed99", "instance_LSTM_seed99")
FIG_MODELS_EXAMPLES = None
PAPER_MODELS = ("stead_LSTM_seed99", "pretrained_LSTM")
PAPER_N_EXAMPLES = 5
PAPER_WINDOW_S = (40.0, 110.0)
PAPER_TIMES = None
PAPER_CFT_YMAX = 20.0

COMMIT = "afe2536cf29ead8b4d8d5c50ddf85cc1dd90ddd1"
STEAD_FILE = "training_datasets_all_snr_40_unshuffled.hdf5"
INSTANCE_FILE = "training_datasets_INSTANCE_snr40.hdf5"
PAPER_BOTTLENECKS = ("None", "Linear", "LSTM", "attention", "Transformer")
NEW_BOTTLENECKS = ("attention_PE", "attention_LSTM", "hybrid", "Transformer_fixed")
print("switches:", {k: v for k, v in sorted(globals().items()) if k.startswith("RUN_") and isinstance(v, bool)})

switches: {'RUN_DOWNLOAD': True, 'RUN_FIGURES': True, 'RUN_PAPER_FIGS': True, 'RUN_REAL': True, 'RUN_SELFTEST': True, 'RUN_SYNTH': True}


In [ ]:
import os, sys, glob, json, time, re, shutil, subprocess, platform, datetime, hashlib, traceback
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle/input")
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
TEMP = Path("/kaggle/temp") if ON_KAGGLE else (Path.cwd() / "tmp")
RESULTS, FIGURES, LOGS, CONT = WORK / "results_08", WORK / "figures_08", WORK / "logs", WORK / "continuous"
for d in (RESULTS, FIGURES, LOGS, CONT, TEMP):
    d.mkdir(parents=True, exist_ok=True)
REPO = TEMP / "WaveDecompNet-paper"
SEARCH_ROOTS = ([Path("/kaggle/input")] if ON_KAGGLE else []) + [WORK]
if MODELS_DIR:
    SEARCH_ROOTS.append(Path(MODELS_DIR))

def log(msg):
    line = "- %s  [08] %s" % (datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), msg)
    with open(LOGS / "lab_log.md", "a") as f:
        f.write(line + "\n")
    print(line)

def find_all(pattern, roots=None):
    out = []
    for r in (roots if roots is not None else SEARCH_ROOTS):
        if Path(r).exists():
            out += [Path(p) for p in glob.glob(str(Path(r) / "**" / pattern), recursive=True)]
    return sorted(set(p for p in out if not p.name.startswith("._")))

def find_one(name, required=True, roots=None):
    hits = find_all(name, roots)
    if not hits:
        if required:
            raise FileNotFoundError("%s not found under %s: attach the dataset that holds it" % (name, [str(r) for r in (roots or SEARCH_ROOTS)]))
        return None
    if len(hits) > 1:
        print("note: %d copies of %s, using %s" % (len(hits), name, hits[0]))
    return hits[0]

def ensure(pip_name, module=None):
    try:
        __import__(module or pip_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
for p, m in (("obspy", None), ("h5py", None), ("scikit-learn", "sklearn"), ("pandas", None)):
    ensure(p, m)

import numpy as np, pandas as pd, h5py, obspy
import torch
print("python", platform.python_version(), "| numpy", np.__version__, "| torch", torch.__version__, "| obspy", obspy.__version__)
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if not (REPO / "autoencoder_1D_models_torch.py").exists():
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.run(["git", "clone", "--quiet", "https://github.com/yinjiuxun/WaveDecompNet-paper.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", COMMIT], check=True)
head = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert head == COMMIT, "repo is at %s, expected %s" % (head, COMMIT)
tt = REPO / "torch_tools.py"
if "np.Inf" in tt.read_text():
    tt.write_text(tt.read_text().replace("np.Inf", "np.inf"))
for p in (REPO, WORK):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
try:
    torch.backends.mha.set_fastpath_enabled(False)
except Exception:
    pass
print("repo commit", head)

python 3.13.16 | numpy 2.1.3 | torch 2.11.0+cu130 | obspy 1.5.2
cuda: True Tesla T4
repo commit afe2536cf29ead8b4d8d5c50ddf85cc1dd90ddd1


In [ ]:
# wdn_model_ext.py from the repository
MODEL_EXT_SRC = r'''"""Extension bottlenecks for WaveDecompNet.

Repo: yinjiuxun/WaveDecompNet-paper @ afe2536cf29ead8b4d8d5c50ddf85cc1dd90ddd1

This module adds new bottleneck variants WITHOUT touching
`autoencoder_1D_models_torch.py`, so the Phase 1 / Phase 2 reproduction path
stays byte-identical.

Every bottleneck here is a plain `nn.Module` taking and returning a tensor of
shape (batch, num_steps, num_features) = (B, 75, 64). `SeismogramDecoder.forward`
therefore picks them up through its generic `else: x = self.bottleneck(x)`
branch with no change to the repo file. None of these classes subclasses
`nn.LSTM` or `nn.MultiheadAttention`, which would divert the dispatch.

Variants
--------
attention_PE
    The paper's `attention` bottleneck plus a sinusoidal positional encoding.
    `nn.MultiheadAttention` with query=key=value and no positional information
    is permutation-invariant along time, so the paper's attention bottleneck
    cannot use ordering; position reaches the decoder only through the
    `x + x3` skip connection. The encoding is parameter-free, so this variant
    has EXACTLY the same parameter count as `attention` (111,370) and any
    change in EV cannot be attributed to added capacity.

attention_LSTM
    One bidirectional LSTM layer used as a learned positional embedding,
    followed by multi-head self-attention. This is the repo's own
    `Attention_bottleneck_LSTM`, which `step3` can already select by name but
    which the paper does not report. Reimplemented on `nn.MultiheadAttention`
    to avoid the softmax-axis defect in the repo's hand-rolled
    `DotProductAttention`: `F.softmax(scores, dim=0)` normalises over the
    BATCH axis instead of over keys, so attention weights do not sum to one
    over keys and, at batch size 1, every weight becomes 1.0.

hybrid
    LSTM bottleneck on the earthquake decoder, attention+PE bottleneck on the
    noise decoder. Also already selectable in `step3` and not reported.

Transformer_fixed
    The paper's `Transformer` bottleneck with two defects corrected:
      * `batch_first=True`. The original passes no `batch_first`, so it
        defaults to False and `nn.TransformerEncoder` reads the input
        (batch, steps, features) as (seq, batch, feature) -- it attends across
        the BATCH axis and never along time.
      * `dim_feedforward=256` instead of the 2048 default, a 4x expansion on a
        64-dim latent rather than 32x. The 2048 default accounts for 281,152 of
        the original 640,394 parameters.
    `norm_first=True` makes it a pre-LN block, and the same parameter-free
    positional encoding is applied, since a corrected temporal attention is
    otherwise still permutation-invariant.

The paper's five bottleneck names are deliberately NOT served by
`make_bottlenecks`; they raise `ValueError`. There is no path by which this
module can alter a reproduction run.
"""

import copy
import math

import torch
from torch import nn

__all__ = [
    "SinusoidalPositionalEncoding",
    "AttentionPEBottleneck",
    "AttentionLSTMBottleneck",
    "TransformerFixedBottleneck",
    "make_bottlenecks",
    "NEW_BOTTLENECKS",
    "PAPER_BOTTLENECKS",
]

PAPER_BOTTLENECKS = ("None", "Linear", "LSTM", "attention", "Transformer")
NEW_BOTTLENECKS = ("attention_PE", "attention_LSTM", "hybrid", "Transformer_fixed")


class SinusoidalPositionalEncoding(nn.Module):
    """Parameter-free sinusoidal positional encoding, added to the input.

    Unlike the repo's `PositionalEncoding`, the table is registered as a buffer
    (so it moves with `.to(device)` and is carried by `torch.save(model)`) and
    is built in the model dtype (the repo builds a float32 table inside a
    float64 model).
    """

    def __init__(self, d_model=64, max_len=512, dropout=0.0, dtype=torch.float64):
        super().__init__()
        if d_model % 2 != 0:
            raise ValueError("d_model must be even, got %d" % d_model)
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(1, max_len, d_model, dtype=dtype)
        position = torch.arange(max_len, dtype=dtype).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=dtype) * (-math.log(10000.0) / d_model)
        )
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (batch, num_steps, num_features)
        if x.shape[1] > self.pe.shape[1]:
            raise ValueError(
                "sequence length %d exceeds max_len %d" % (x.shape[1], self.pe.shape[1])
            )
        return self.dropout(x + self.pe[:, : x.shape[1], :])


class AttentionPEBottleneck(nn.Module):
    """The paper's `attention` bottleneck + sinusoidal positional encoding.

    Parameter count is identical to the paper's `attention` bottleneck.
    """

    def __init__(self, d_model=64, nhead=4, dropout=0.2, pe_dropout=0.0,
                 max_len=512, dtype=torch.float64):
        super().__init__()
        self.pe = SinusoidalPositionalEncoding(d_model, max_len, pe_dropout, dtype)
        self.attention = nn.MultiheadAttention(
            d_model, nhead, dropout, batch_first=True, dtype=dtype
        )

    def forward(self, x):
        x = self.pe(x)
        x, _ = self.attention(x, x, x)
        return x


class AttentionLSTMBottleneck(nn.Module):
    """Bidirectional LSTM as positional embedding, then self-attention.

    Corrected reimplementation of the repo's `Attention_bottleneck_LSTM`.
    """

    def __init__(self, d_model=64, nhead=4, dropout=0.2, dtype=torch.float64):
        super().__init__()
        if d_model % 2 != 0:
            raise ValueError("d_model must be even, got %d" % d_model)
        self.lstm = nn.LSTM(
            d_model, d_model // 2, 1, bidirectional=True, batch_first=True, dtype=dtype
        )
        self.attention = nn.MultiheadAttention(
            d_model, nhead, dropout, batch_first=True, dtype=dtype
        )

    def forward(self, x):
        x, _ = self.lstm(x)
        x, _ = self.attention(x, x, x)
        return x


class TransformerFixedBottleneck(nn.Module):
    """Pre-LN transformer encoder block that actually attends along time."""

    def __init__(self, d_model=64, nhead=4, dim_feedforward=256, dropout=0.1,
                 num_layers=1, use_pe=True, max_len=512, dtype=torch.float64):
        super().__init__()
        self.pe = (
            SinusoidalPositionalEncoding(d_model, max_len, 0.0, dtype) if use_pe else None
        )
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            dtype=dtype,
        )
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=num_layers, enable_nested_tensor=False
        )

    def forward(self, x):
        if self.pe is not None:
            x = self.pe(x)
        return self.encoder(x)


def _pair(module):
    """Two bottlenecks with identical initial weights.

    `step3` builds one bottleneck and `copy.deepcopy`s it for the two decoders,
    so both branches start from the same bottleneck weights. That behaviour is
    kept here so the only difference from the paper's runs is the bottleneck
    type.
    """
    return module, copy.deepcopy(module)


def make_bottlenecks(name, d_model=64, nhead=4, dropout=0.2, dtype=torch.float64):
    """Return (bottleneck_earthquake, bottleneck_noise) for an extension variant.

    Raises ValueError for the paper's five bottleneck names, which must stay on
    the original `step3` code path.
    """
    if name in PAPER_BOTTLENECKS:
        raise ValueError(
            "'%s' is one of the paper's bottlenecks and must be built by the "
            "original step3 code, not by wdn_model_ext." % name
        )

    if name == "attention_PE":
        return _pair(AttentionPEBottleneck(d_model, nhead, dropout, dtype=dtype))

    if name == "attention_LSTM":
        return _pair(AttentionLSTMBottleneck(d_model, nhead, dropout, dtype=dtype))

    if name == "Transformer_fixed":
        return _pair(TransformerFixedBottleneck(d_model, nhead, dtype=dtype))

    if name == "hybrid":
        # Repo intent (step3 'hybrid'): LSTM for earthquake, attention for noise.
        # The LSTM matches the paper's `LSTM` bottleneck (2 layers, bidirectional).
        bottleneck_earthquake = nn.LSTM(
            d_model, d_model // 2, 2, bidirectional=True, batch_first=True, dtype=dtype
        )
        bottleneck_noise = AttentionPEBottleneck(d_model, nhead, dropout, dtype=dtype)
        return bottleneck_earthquake, bottleneck_noise

    raise NameError(
        "unknown bottleneck '%s'; extension variants are %s" % (name, list(NEW_BOTTLENECKS))
    )
'''
(REPO / "wdn_model_ext.py").write_text(MODEL_EXT_SRC)
import autoencoder_1D_models_torch, wdn_model_ext   # both must be importable before torch.load
print("wdn_model_ext.py sha256[:16] =", hashlib.sha256(MODEL_EXT_SRC.encode()).hexdigest()[:16])
log("Part 0: repo %s, device %s, NORM=%s, OVERLAP=%s" % (head[:10], DEVICE, NORM, OVERLAP))

wdn_model_ext.py sha256[:16] = 65dea6e1eb352830
- 2026-11-23 12:19:50  [08] Part 0: repo afe2536cf2, device cuda:0, NORM=window, OVERLAP=0.5


In [ ]:
# Helper functions: models
PREFIX = "Branch_Encoder_Decoder_"
FS, WIN = 10.0, 600
GROUP_ORDER = ("stead", "instance", "variant", "pretrained")

def parse_run_folder(name):
    if not name.startswith(PREFIX):
        return None
    rest = name[len(PREFIX):]
    kind = "smoke" if "_smoke" in rest else "full"
    trained_on = "INSTANCE" if "_instance" in rest else "STEAD"
    b, seed = rest, 99
    m = re.search(r"_seed(\d+)$", b)
    if m:
        seed, b = int(m.group(1)), b[:m.start()]
    for suf in ("_instance", "_warmup", "_smoke"):
        b = b.replace(suf, "")
    return b, trained_on, seed, kind

def discover_models():
    out = {}
    for pth in find_all("*_Model.pth"):
        if REPO in pth.parents:
            continue
        parsed = parse_run_folder(pth.parent.name)
        if parsed is None or parsed[3] == "smoke":
            continue
        b, on, seed, _ = parsed
        info = {}
        if (pth.parent / "run_info.json").exists():
            info = json.loads((pth.parent / "run_info.json").read_text())
        if info.get("subset") is not None or (EPOCHS_REQUIRED and info.get("epochs") not in (None, EPOCHS_REQUIRED)):
            print("skip (subset or epochs):", pth.parent)
            continue
        group = "variant" if b in NEW_BOTTLENECKS else on.lower()
        mid = "%s_%s_seed%d" % (on.lower(), b, seed)
        if mid in out:
            print("duplicate %s, keeping %s, ignoring %s" % (mid, out[mid]["path"].parent, pth.parent))
            continue
        out[mid] = dict(model_id=mid, bottleneck=b, trained_on=on, group=group, kind="new" if b in NEW_BOTTLENECKS else "paper",
                        seed=seed, path=pth, n_params_run=info.get("n_params"))
    if INCLUDE_PRETRAINED:
        for b in PAPER_BOTTLENECKS:
            hits = sorted(REPO.glob("trained_models/%s%s/%s%s_Model.pth" % (PREFIX, b, PREFIX, b))) or sorted(REPO.glob("**/%s%s_Model.pth" % (PREFIX, b)))
            if hits:
                out["pretrained_%s" % b] = dict(model_id="pretrained_%s" % b, bottleneck=b, trained_on="AUTHORS", group="pretrained", kind="pretrained",
                                                seed=None, path=hits[0], n_params_run=None)
            else:
                print("pretrained %s not found in the repo" % b)
    if ONLY_MODELS:
        out = {k: v for k, v in out.items() if k in ONLY_MODELS}
    order = lambda k: (GROUP_ORDER.index(out[k]["group"]), (PAPER_BOTTLENECKS + NEW_BOTTLENECKS).index(out[k]["bottleneck"]))
    return {k: out[k] for k in sorted(out, key=order)}

COMPAT_DEFAULTS = {"norm_first": False, "activation_relu_or_gelu": 1, "enable_nested_tensor": False, "use_nested_tensor": False, "mask_check": True}
COMPAT_LOG = []

def check_forward(model):
    x = torch.randn(2, 3, WIN, dtype=torch.float64, device=DEVICE)
    for _ in range(8):
        try:
            with torch.no_grad():
                o = model(x)
            assert len(o) == 2 and tuple(o[0].shape) == (2, 3, WIN) and tuple(o[1].shape) == (2, 3, WIN), "unexpected output shapes"
            return True, None
        except AttributeError as e:
            name = (re.search(r"has no attribute '(\w+)'", str(e)) or [None, None])[1]
            if name not in COMPAT_DEFAULTS:
                return False, traceback.format_exc()
            for mod in model.modules():
                if type(mod).__name__ in ("TransformerEncoderLayer", "TransformerEncoder") and not hasattr(mod, name):
                    setattr(mod, name, COMPAT_DEFAULTS[name])
            COMPAT_LOG.append(name)
        except Exception:
            return False, traceback.format_exc()
    return False, "compat shim did not converge: " + ", ".join(COMPAT_LOG)

def load_model(path):
    m = torch.load(str(path), map_location=DEVICE, weights_only=False)
    m = m.to(DEVICE).double().eval()
    ok, err = check_forward(m)
    assert ok, err
    return m

def n_params(m):
    return int(sum(p.numel() for p in m.parameters()))

def model_predictor(model):
    """(B, 3, 600) float64 normalised -> (earthquake, noise) as numpy, fixed batch size BATCH."""
    def f(xn):
        pe, pn = np.empty_like(xn), np.empty_like(xn)
        with torch.no_grad():
            for i in range(0, len(xn), BATCH):
                o1, o2 = model(torch.from_numpy(xn[i:i + BATCH]).to(DEVICE))
                pe[i:i + BATCH], pn[i:i + BATCH] = o1.cpu().numpy(), o2.cpu().numpy()
        return pe, pn
    return f

In [ ]:
# Helper functions: separation of a long stream and STA/LTA
from obspy.signal.trigger import recursive_sta_lta, trigger_onset

def window_starts(n, hop):
    if n < WIN:
        return np.zeros(0, int)
    s = np.arange(0, n - WIN + 1, hop)
    return s if s[-1] == n - WIN else np.append(s, n - WIN)

def separate_stream(pred, w, norm=None, overlap=None, gstats=None, chunk=2048):
    """Slide the model over a long 3-component stream w (n, 3) at 10 Hz.

    Each 60 s window is normalised (per window, or with one global mean/std), separated, de-normalised and
    blended with its neighbours with a periodic Hann taper (sums to one at 50 % overlap). Single-covered samples
    are divided by their own weight, so the result is exact where only one window contributes.
    Returns earthquake, noise (n, 3) in the units of w.
    """
    norm = NORM if norm is None else norm
    overlap = OVERLAP if overlap is None else overlap
    n = len(w)
    hop = int(round(WIN * (1 - overlap)))
    assert 0 < hop <= WIN
    starts = window_starts(n, hop)
    tap = np.ones(WIN) if hop == WIN else np.maximum(0.5 - 0.5 * np.cos(2 * np.pi * np.arange(WIN) / WIN), 1e-3)
    eq, nz, ws = np.zeros((n, 3)), np.zeros((n, 3)), np.zeros(n)
    if norm == "global":
        mu_g, sd_g = gstats if gstats is not None else (w.mean(0), w.std(0) + 1e-12)
    ar = np.arange(WIN)
    for i in range(0, len(starts), chunk):
        ch = starts[i:i + chunk]
        seg = w[ch[:, None] + ar[None, :]]
        if norm == "window":
            mu, sd = seg.mean(1, keepdims=True), seg.std(1, keepdims=True) + 1e-12
        else:
            mu, sd = mu_g, sd_g
        xn = np.ascontiguousarray(np.moveaxis((seg - mu) / sd, 1, -1))
        pe, pn = pred(xn)
        pe = np.moveaxis(pe, 1, -1) * sd
        pn = np.moveaxis(pn, 1, -1) * sd + mu
        for k, s0 in enumerate(ch):
            eq[s0:s0 + WIN] += pe[k] * tap[:, None]
            nz[s0:s0 + WIN] += pn[k] * tap[:, None]
            ws[s0:s0 + WIN] += tap
    ok = ws > 0
    eq[ok] /= ws[ok, None]
    nz[ok] /= ws[ok, None]
    return eq, nz

def cft3(x, fs=FS):
    return [recursive_sta_lta(np.ascontiguousarray(x[:, c], dtype=np.float64), int(STA_S * fs), int(LTA_S * fs)) for c in range(x.shape[1])]

def coincidence(cfts, thr_on, thr_off, thr_sum=COINCIDENCE):
    """obspy.signal.trigger.coincidence_trigger, re-implemented on sample indices with weight 1 per component.

    Same merging rules (overlapping triggers of other components extend `off`; a trigger whose off is not later
    than the last accepted off is a subset and is skipped). Lets the threshold sweep reuse one cft per component.
    Part 1 checks it against ObsPy. Returns (n, 2) int array of [on, off] samples. """
    trig = []
    for c, cft in enumerate(cfts):
        for a, b in trigger_onset(cft, thr_on, thr_off):
            trig.append((int(a), int(b), c))
    trig.sort()
    out, last_off = [], -np.inf
    for i, (on, off, c) in enumerate(trig):
        comps, s = {c}, 1
        for on2, off2, c2 in trig[i + 1:]:
            if c2 in comps:
                continue
            if on2 > off:
                break
            comps.add(c2); s += 1; off = max(off, off2)
        if s < thr_sum or off <= last_off:
            continue
        out.append((on, off)); last_off = off
    return np.array(out, dtype=int).reshape(-1, 2)

def sweep_triggers(x, valid=None):
    cf = cft3(x)
    out = {}
    for thr in TRIG_ON_SWEEP:
        t = coincidence(cf, thr, min(TRIG_OFF, thr - 0.5))[:, 0]
        if valid is not None and len(t):
            t = t[valid[t]]
        out[thr] = t
    return out

BASELINES = ("raw", "bandpass")

def bandpass(x):
    from scipy.signal import butter, sosfiltfilt
    sos = butter(4, BANDPASS_HZ, btype="bandpass", fs=FS, output="sos")
    return sosfiltfilt(sos, x, axis=0)

def baseline_or_model(mid, x, pred, **kw):
    if mid == "raw":
        return x, np.zeros_like(x)
    if mid == "bandpass":
        e = bandpass(x)
        return e, x - e
    return separate_stream(pred, x, **kw)

def ev_rows(y, p):
    num, den = (y - p).var(0), y.var(0)
    ev = np.where(den > 0, 1 - num / np.maximum(den, 1e-300), 0.0)
    return float(np.clip(ev, -1, 1).mean())

def rms(a):
    return float(np.sqrt(np.mean(np.square(a)))) if a.size else np.nan

## Part 1 : Self-test

In [6]:
MODELS = discover_models()
rows = []
for mid, m in MODELS.items():
    try:
        mod = load_model(m["path"])
        m["n_params"] = n_params(mod)
        status = "ok" if (m["n_params_run"] in (None, m["n_params"])) else "PARAM MISMATCH (run_info %s)" % m["n_params_run"]
        del mod
    except Exception as e:
        m["n_params"], status = None, "LOAD FAILED: %s" % str(e).splitlines()[-1][:120]
    rows.append(dict(model_id=mid, group=m["group"], kind=m["kind"], bottleneck=m["bottleneck"], trained_on=m["trained_on"], n_params=m["n_params"], status=status, path=str(m["path"])))
MODEL_TABLE = pd.DataFrame(rows)
MODEL_TABLE.to_csv(RESULTS / "model_table.csv", index=False)
print(MODEL_TABLE[["model_id", "kind", "trained_on", "n_params", "status"]].to_string(index=False))
bad = MODEL_TABLE[MODEL_TABLE.status != "ok"]
if len(bad):
    print("\nWARNING: these models will be skipped:\n", bad[["model_id", "status"]].to_string(index=False))
MODELS = {k: v for k, v in MODELS.items() if k in set(MODEL_TABLE[MODEL_TABLE.status == "ok"].model_id)}
if COMPAT_LOG:
    print("compat shim used for:", sorted(set(COMPAT_LOG)))
log("Part 1: %d models usable: %s" % (len(MODELS), ", ".join(MODELS)))

                      model_id       kind trained_on  n_params status
             stead_None_seed99      paper      STEAD     78090     ok
           stead_Linear_seed99      paper      STEAD     86410     ok
             stead_LSTM_seed99      paper      STEAD    178442     ok
        stead_attention_seed99      paper      STEAD    111370     ok
      stead_Transformer_seed99      paper      STEAD    640394     ok
          instance_None_seed99      paper   INSTANCE     78090     ok
        instance_Linear_seed99      paper   INSTANCE     86410     ok
          instance_LSTM_seed99      paper   INSTANCE    178442     ok
     instance_attention_seed99      paper   INSTANCE    111370     ok
   instance_Transformer_seed99      paper   INSTANCE    640394     ok
     stead_attention_PE_seed99        new      STEAD    111370     ok
   stead_attention_LSTM_seed99        new      STEAD    161546     ok
           stead_hybrid_seed99        new      STEAD    144906     ok
stead_Transformer_fi

In [ ]:
if RUN_SELFTEST:
    from obspy import Stream, Trace, UTCDateTime
    from obspy.signal.trigger import coincidence_trigger
    rng = np.random.default_rng(0)
    n = int(4 * 3600 * FS)
    x = rng.standard_normal((n, 3))
    for t0 in rng.integers(1000, n - 2000, 60):
        for c in range(3):
            if rng.random() < 0.7:
                L = int(rng.integers(20, 600))
                x[t0 + c * 7: t0 + c * 7 + L, c] *= rng.uniform(1, 30)
    st = Stream([Trace(np.ascontiguousarray(x[:, c]), header=dict(network="XX", station="SYN", channel="HH" + "ENZ"[c], sampling_rate=FS, starttime=UTCDateTime(0))) for c in range(3)])
    for thr in (3.0, TRIG_ON):
        ref = coincidence_trigger("recstalta", thr, TRIG_OFF, st.copy(), COINCIDENCE, sta=STA_S, lta=LTA_S)
        ref_on = np.array([int(round(float(t["time"] - UTCDateTime(0)) * FS)) for t in ref])
        ours = coincidence(cft3(x), thr, TRIG_OFF)[:, 0]
        assert np.array_equal(ref_on, ours), "coincidence trigger differs from ObsPy at thr_on=%s: %d vs %d" % (thr, len(ref_on), len(ours))
        print("coincidence == ObsPy at thr_on=%.1f: %d triggers" % (thr, len(ours)))

    ident = lambda xn: (xn.copy(), np.zeros_like(xn))
    w = rng.standard_normal((WIN * 50 + 137, 3)) * [3, 1, 2] + [5, -1, 0]
    for norm in ("window", "global"):
        for ov in (0.0, 0.5):
            eq, nz = separate_stream(ident, w, norm=norm, overlap=ov)
            mu = w.mean(0) if norm == "global" else None
            assert np.allclose(eq + nz, w, atol=1e-9), (norm, ov)
            print("stitching exact: norm=%-6s overlap=%.1f  max|eq+nz-w| = %.1e" % (norm, ov, np.abs(eq + nz - w).max()))
    log("Part 1: self-test passed")
else:
    print("Part 1 self-test skipped")

coincidence == ObsPy at thr_on=3.0: 80 triggers
coincidence == ObsPy at thr_on=6.0: 52 triggers
stitching exact: norm=window overlap=0.0  max|eq+nz-w| = 3.6e-15
stitching exact: norm=window overlap=0.5  max|eq+nz-w| = 3.6e-15
stitching exact: norm=global overlap=0.0  max|eq+nz-w| = 1.8e-15
stitching exact: norm=global overlap=0.5  max|eq+nz-w| = 3.6e-15
- 2026-11-23 12:20:12  [08] Part 1: self-test passed


## Part 2 : Continuous data, catalogue, travel times

In [ ]:
def station_tag(s):
    return "%s.%s" % (s["net"], s["sta"])

def proc_name(s):
    return "%s_%s_%dd_10Hz.npz" % (station_tag(s), START.replace("-", ""), N_DAYS)

def fdsn(name):
    from obspy.clients.fdsn import Client
    return Client(name, timeout=300)

def fetch(fn_name, *args, **kw):
    from obspy.clients.fdsn.header import FDSNNoDataException
    last = None
    for attempt in range(3):
        for cname in (FDSN_CLIENTS if fn_name != "get_events" else ("USGS",)):
            try:
                return getattr(fdsn(cname), fn_name)(*args, **kw)
            except FDSNNoDataException:
                return None
            except Exception as e:
                last = e
        time.sleep(15 * (attempt + 1))
    raise RuntimeError("%s failed after retries: %s" % (fn_name, last))

def component_order(st):
    codes = sorted({tr.stats.channel for tr in st}, key=lambda c: (c[-1] == "Z", c))
    assert len(codes) == 3, "expected 3 channels, got %s" % codes
    return codes

def day_to_10hz(st, t0, pad, inv=None):
    from scipy import signal as sgn
    from scipy.interpolate import interp1d
    locs = sorted({tr.stats.location for tr in st})
    st = st.select(location=locs[0])
    st.merge(method=1, fill_value=None)
    if ROTATE_TO_ZNE and inv is not None:
        st.rotate("->ZNE", inventory=inv)
    codes = component_order(st)
    n_out = int(round(86400 * FS))
    grid = np.arange(-pad * FS, (86400 + pad) * FS) / FS
    w = np.zeros((len(grid), 3)); good = np.ones(len(grid), bool)
    for c, code in enumerate(codes):
        sel = st.select(channel=code)
        if len(sel) == 0:
            good[:] = False
            continue
        tr = sel[0]
        fs0 = tr.stats.sampling_rate
        d = np.ma.masked_invalid(np.ma.asarray(tr.data, dtype=float))
        m = np.ma.getmaskarray(d)
        d = d.filled(np.ma.median(d) if (~m).any() else 0.0)
        b, a = sgn.butter(4, FS / 2 * 2 / fs0)
        d = sgn.filtfilt(b, a, d - d.mean())
        t = (tr.stats.starttime - t0) + np.arange(tr.stats.npts) / fs0
        w[:, c] = interp1d(t, d, bounds_error=False, fill_value=0.0)(grid)
        gap = interp1d(t, m.astype(float), kind="nearest", bounds_error=False, fill_value=1.0)(grid) > 0
        good &= ~gap
    i0 = int(round(pad * FS))
    w, good = w[i0:i0 + n_out], good[i0:i0 + n_out]
    if good.any():
        w -= np.median(w[good], axis=0)
    w[~good] = 0.0
    return w, good, codes

def load_or_download_station(s):
    from obspy import UTCDateTime, read
    tag = station_tag(s)
    hit = find_one(proc_name(s), required=False, roots=SEARCH_ROOTS + [CONT])
    if hit is not None:
        z = np.load(hit, allow_pickle=True)
        print("%s: using %s" % (tag, hit))
        return z["w"].astype(np.float64), z["good"], json.loads(str(z["meta"]))
    t_start, pad = UTCDateTime(START), 300.0
    inv = fetch("get_stations", network=s["net"], station=s["sta"], location=s["loc"], channel=s["cha"],starttime=t_start, endtime=t_start + N_DAYS * 86400, level="response")
    inv.write(str(CONT / ("%s_inventory.xml" % tag)), format="STATIONXML")
    W, G, codes = [], [], None
    rawdir = CONT / "raw" / tag; rawdir.mkdir(parents=True, exist_ok=True)
    for k in range(N_DAYS):
        t0 = t_start + k * 86400
        f = rawdir / ("%s.mseed" % t0.strftime("%Y%m%d"))
        t1 = time.time()
        if f.exists():
            st = read(str(f))
        else:
            st = fetch("get_waveforms", s["net"], s["sta"], s["loc"], s["cha"], t0 - pad, t0 + 86400 + pad)
            if st is not None and len(st):
                st.write(str(f), format="MSEED")
        if st is None or len(st) == 0:
            print("  %s %s: no data" % (tag, t0.date)); W.append(np.zeros((int(86400 * FS), 3))); G.append(np.zeros(int(86400 * FS), bool)); continue
        w, g, c = day_to_10hz(st, t0, pad, inv)
        codes = codes or c
        W.append(w); G.append(g)
        print("  %s %s: %d traces, %.1f %% valid, %.0f s" % (tag, t0.date, len(st), 100 * g.mean(), time.time() - t1))
        if not KEEP_RAW_MSEED:
            f.unlink(missing_ok=True)
    w, good = np.concatenate(W), np.concatenate(G)
    sta = inv[0][0]
    meta = dict(station=tag, start=str(t_start), n_days=N_DAYS, fs=FS, channels=codes, lat=sta.latitude, lon=sta.longitude, elevation=sta.elevation, rotated=ROTATE_TO_ZNE)
    np.savez_compressed(CONT / proc_name(s), w=w.astype(np.float32), good=good, meta=json.dumps(meta))
    return w, good, meta

def catalogue_for(meta):
    from obspy import UTCDateTime, read_events
    from obspy.geodetics import locations2degrees
    from obspy.taup import TauPyModel
    t_start = UTCDateTime(meta["start"]); t_end = t_start + meta["n_days"] * 86400
    if CATALOG == "repo":
        p = REPO / "notebooks" / "data_and_model" / "catalog_local.20210731-20210901.xml"
        assert p.exists(), p
        assert UTCDateTime("2021-07-31") <= t_start and t_end <= UTCDateTime("2021-09-01"), "repo catalogue covers 2021-07-31..09-01 only; use CATALOG='usgs'"
        cat = read_events(str(p))
    else:
        cat = fetch("get_events", starttime=t_start - 600, endtime=t_end, latitude=meta["lat"], longitude=meta["lon"],
                    maxradius=CATALOG_RADIUS_DEG, minmagnitude=CATALOG_MIN_MAG)
    tau = TauPyModel("iasp91"); rows = []
    for ev in cat:
        try:
            o = ev.preferred_origin() or ev.origins[0]
            mg = ev.preferred_magnitude() or ev.magnitudes[0]
        except Exception:
            continue
        if mg.mag is None or mg.mag < CATALOG_MIN_MAG or not (t_start - 600 <= o.time <= t_end):
            continue
        dist = locations2degrees(meta["lat"], meta["lon"], o.latitude, o.longitude)
        dep = max((o.depth or 10e3) / 1e3, 0.0)
        tp = [a.time for a in tau.get_travel_times(dep, dist, phase_list=["p", "P"])]
        ts = [a.time for a in tau.get_travel_times(dep, dist, phase_list=["s", "S"])]
        if not tp or not ts:
            continue
        t0 = o.time - t_start
        rows.append(dict(origin=str(o.time), lat=o.latitude, lon=o.longitude, depth_km=dep, dist_deg=dist, mag=float(mg.mag), mag_type=mg.magnitude_type, tP=t0 + min(tp), tS=t0 + min(ts)))
    df = pd.DataFrame(rows).sort_values("tP").reset_index(drop=True)
    return df[(df.tP >= 0) & (df.tP < meta["n_days"] * 86400)].reset_index(drop=True)

def valid_mask(good):
    from scipy.ndimage import binary_erosion
    v = binary_erosion(good, structure=np.ones(2 * int(LTA_S * FS) + 1, bool), border_value=False)
    v[: int(LTA_S * FS)] = False
    return v

STREAMS = {}

def prepare_streams():
    for s in STATIONS:
        tag = station_tag(s)
        w, good, meta = load_or_download_station(s)
        ev = catalogue_for(meta)
        valid = valid_mask(good)
        idx = np.clip((ev.tP.values * FS).astype(int), 0, len(valid) - 1)
        ev["usable"] = valid[idx] & valid[np.clip(idx + int(60 * FS), 0, len(valid) - 1)]
        ev.to_csv(RESULTS / ("catalogue_%s.csv" % tag), index=False)
        STREAMS[tag] = dict(w=w, good=good, valid=valid, meta=meta, events=ev)
        print("%s: %.1f days, %.1f %% valid, channels %s, %d catalogue events (%d usable), M %.1f..%.1f"
              % (tag, len(w) / FS / 86400, 100 * valid.mean(), meta["channels"], len(ev), ev.usable.sum(), ev.mag.min(), ev.mag.max()))
        log("Part 2: %s %s+%dd, %.1f %% valid, %d usable events" % (tag, START, N_DAYS, 100 * valid.mean(), ev.usable.sum()))

if RUN_DOWNLOAD or RUN_REAL:
    prepare_streams()
else:
    print("Part 2 skipped")

  HV.HAT 2021-08-02: 3 traces, 100.0 % valid, 6 s
  HV.HAT 2021-08-03: 3 traces, 100.0 % valid, 6 s
  HV.HAT 2021-08-04: 3 traces, 100.0 % valid, 6 s
  HV.HAT 2021-08-05: 3 traces, 100.0 % valid, 6 s
  HV.HAT 2021-08-06: 3 traces, 100.0 % valid, 6 s
  HV.HAT 2021-08-07: 3 traces, 100.0 % valid, 6 s
  HV.HAT 2021-08-08: 3 traces, 100.0 % valid, 6 s
  HV.HAT 2021-08-09: 3 traces, 100.0 % valid, 6 s
  HV.HAT 2021-08-10: 3 traces, 100.0 % valid, 6 s
  HV.HAT 2021-08-11: 3 traces, 100.0 % valid, 6 s
  HV.HAT 2021-08-12: 3 traces, 100.0 % valid, 6 s
  HV.HAT 2021-08-13: 6 traces, 99.4 % valid, 6 s
  HV.HAT 2021-08-14: 3 traces, 100.0 % valid, 6 s
  HV.HAT 2021-08-15: 3 traces, 100.0 % valid, 6 s
HV.HAT: 14.0 days, 99.9 % valid, channels ['HHE', 'HHN', 'HHZ'], 483 catalogue events (483 usable), M 1.5..4.0
- 2026-11-23 12:22:28  [08] Part 2: HV.HAT 2021-08-02+14d, 99.9 % valid, 483 usable events
  IU.POHA 2021-08-02: 3 traces, 100.0 % valid, 2 s
  IU.POHA 2021-08-03: 3 traces, 100.0 % valid, 2

## Part 3 : Real data, every model

In [ ]:
def score_real(trig, ev, valid):
    t = np.sort(trig) / FS
    u = ev[ev.usable]
    lo, hi = np.searchsorted(t, u.tP.values - MATCH_PRE), np.searchsorted(t, u.tP.values + MATCH_POST, side="right")
    det = hi > lo
    a, b = np.searchsorted(t, ev.tP.values - MATCH_PRE), np.searchsorted(t, ev.tS.values + EVENT_TAIL, side="right")
    belongs = np.zeros(len(t), bool)
    for i, j in zip(a, b):
        belongs[i:j] = True
    return det, int((~belongs).sum()), valid.sum() / FS / 3600

def example_events(ev):
    u = ev[ev.usable].reset_index(drop=True)
    if not len(u):
        return u
    pick = [int(u.mag.idxmax())]
    for m in (1.0, 1.5, 2.0, 2.5, 3.0):
        k = int((u.mag - m).abs().idxmin())
        if k not in pick:
            pick.append(k)
    return u.loc[pick[:N_EXAMPLE_EVENTS]].sort_values("mag")

def real_one(tag, mid, pred):
    S = STREAMS[tag]; w, valid, ev = S["w"], S["valid"], S["events"]
    t1 = time.time()
    gstats = (w[S["good"]].mean(0), w[S["good"]].std(0) + 1e-12)
    eq, nz = baseline_or_model(mid, w, pred, gstats=gstats)
    t_sep = time.time() - t1
    trig = sweep_triggers(eq, valid)
    res = dict(model_id=mid, station=tag, channels=S["meta"]["channels"], seconds_separate=t_sep, norm=NORM, overlap=OVERLAP, sweep={})
    for thr, t in trig.items():
        det, n_unc, hours = score_real(t, ev, valid)
        u = ev[ev.usable]
        by_mag = {"%g-%g" % (lo, hi): (float(det[(u.mag >= lo).values & (u.mag < hi).values].mean()) if ((u.mag >= lo) & (u.mag < hi)).any() else None)
                  for lo, hi in zip(MAG_BINS[:-1], MAG_BINS[1:])}
        res["sweep"][str(thr)] = dict(n_trig=int(len(t)), recall=float(det.mean()) if len(det) else None,
                                      recall_m2=float(det[(u.mag >= 2).values].mean()) if (u.mag >= 2).any() else None,
                                      uncat_per_h=n_unc / hours, recall_by_mag=by_mag, detected=det.astype(int).tolist())
    u = ev[ev.usable]
    gains = []
    for tp in u.tP.values:
        i = int(tp * FS)
        sig_r, pre_r = w[i:i + 100], w[i - 350:i - 50]
        sig_e, pre_e = eq[i:i + 100], eq[i - 350:i - 50]
        floor = 1e-3 * rms(pre_r)
        gains.append(20 * np.log10(max(rms(sig_e), floor) / max(rms(pre_e), floor)) - 20 * np.log10(max(rms(sig_r), floor) / max(rms(pre_r), floor)))
    res["snr_gain_db_median"] = float(np.median(gains)) if gains else None
    res["snr_gain_db"] = [float(g) for g in gains]
    nh = int(len(w) // int(3600 * FS)); leak = []
    busy = np.zeros(nh, bool)
    for tp, ts in zip(ev.tP.values, ev.tS.values):
        busy[max(int((tp - 120) // 3600), 0):min(int((ts + 120) // 3600), nh - 1) + 1] = True
    for h in range(nh):
        sl = slice(h * int(3600 * FS), (h + 1) * int(3600 * FS))
        if not busy[h] and valid[sl].mean() > 0.9:
            leak.append(rms(eq[sl][valid[sl]]) / max(rms(w[sl][valid[sl]]), 1e-12))
    res["quiet_leak_median"] = float(np.median(leak)) if leak else None
    res["residual"] = rms((w - eq - nz)[valid]) / max(rms(w[valid]), 1e-12)
    t6 = trig[TRIG_ON]
    res["boundary_frac"] = float(np.mean(np.minimum(t6 % WIN, WIN - t6 % WIN) <= FS)) if len(t6) else None
    res["hourly"] = np.bincount((t6 // int(3600 * FS)).astype(int), minlength=nh)[:nh].tolist()
    snip = {}
    for _, e in example_events(ev).iterrows():
        i = int(e.tP * FS)
        snip["%.3f" % e.tP] = eq[i - 300:i + 900].astype(np.float32)
    np.savez_compressed(RESULTS / "real" / tag / ("%s_snippets.npz" % mid), **snip)
    np.savez_compressed(RESULTS / "real" / tag / ("%s_triggers.npz" % mid), **{str(k): v for k, v in trig.items()})
    json.dump(res, open(RESULTS / "real" / tag / ("%s.json" % mid), "w"))
    s6 = res["sweep"][str(TRIG_ON)]
    print("  %-26s triggers %6d  recall %.3f  recall(M>=2) %s  uncat/h %6.2f  gain %5.1f dB  leak %.3f  resid %.3f  bnd %.3f  (%.0f s)"
          % (mid, s6["n_trig"], np.nan if s6["recall"] is None else s6["recall"], "%.3f" % s6["recall_m2"] if s6["recall_m2"] is not None else "  -  ", s6["uncat_per_h"],
             np.nan if res["snr_gain_db_median"] is None else res["snr_gain_db_median"], np.nan if res["quiet_leak_median"] is None else res["quiet_leak_median"], res["residual"], res["boundary_frac"] or 0, time.time() - t1))
    return res

if RUN_REAL:
    for tag in STREAMS:
        (RESULTS / "real" / tag).mkdir(parents=True, exist_ok=True)
        print("== %s" % tag)
        for mid in list(BASELINES) + list(MODELS):
            if RESUME and (RESULTS / "real" / tag / ("%s.json" % mid)).exists():
                print("  %-26s done (RESUME)" % mid); continue
            pred = None if mid in BASELINES else model_predictor(load_model(MODELS[mid]["path"]))
            real_one(tag, mid, pred)
            del pred
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        log("Part 3: %s done for %d models" % (tag, len(MODELS)))
else:
    print("Part 3 skipped")

== HV.HAT
  raw                        triggers    190  recall 0.039  recall(M>=2) 0.070  uncat/h   0.50  gain   0.0 dB  leak 1.000  resid 0.000  bnd 0.042  (4 s)
  bandpass                   triggers    788  recall 0.352  recall(M>=2) 0.480  uncat/h   1.83  gain   3.3 dB  leak 0.011  resid 0.000  bnd 0.046  (11 s)
  stead_None_seed99          triggers  12530  recall 0.698  recall(M>=2) 0.799  uncat/h  36.15  gain  14.4 dB  leak 0.004  resid 0.013  bnd 0.040  (119 s)
  stead_Linear_seed99        triggers  13537  recall 0.735  recall(M>=2) 0.817  uncat/h  39.13  gain  14.4 dB  leak 0.004  resid 0.007  bnd 0.038  (140 s)
  stead_LSTM_seed99          triggers  10399  recall 0.652  recall(M>=2) 0.751  uncat/h  29.93  gain  17.9 dB  leak 0.004  resid 0.005  bnd 0.041  (99 s)
  stead_attention_seed99     triggers   9949  recall 0.739  recall(M>=2) 0.808  uncat/h  28.43  gain  19.5 dB  leak 0.001  resid 0.003  bnd 0.034  (74 s)
  stead_Transformer_seed99   triggers  12036  recall 0.727  recal

## Part 4 : Long synthetic streams with known earthquakes


In [ ]:
def stead_split_indices(n):
    """step3's split on row indices: train_size=0.6 / seed 13, then 50/50 / seed 20 (notebook 07)."""
    from sklearn.model_selection import train_test_split
    idx = np.arange(n)
    tr, rest = train_test_split(idx, train_size=0.6, random_state=13)
    va, te = train_test_split(rest, test_size=0.5, random_state=20)
    return tr, va, te

def load_rows(path, rows, block=20000):
    ri = np.sort(np.asarray(rows))
    with h5py.File(path, "r") as f:
        X, Y = f["X_train"], f["Y_train"]
        Xo = np.empty((len(ri),) + X.shape[1:]); Yo = np.empty_like(Xo); k = 0
        for s in range(0, X.shape[0], block):
            m = ri[(ri >= s) & (ri < s + block)]
            if len(m):
                Xo[k:k + len(m)], Yo[k:k + len(m)] = X[s:s + block][m - s], Y[s:s + block][m - s]; k += len(m)
    assert k == len(ri)
    return Xo, Yo

def test_pool(source, rng):
    if source == "STEAD":
        path = find_one(STEAD_FILE)
        with h5py.File(path, "r") as f:
            n = f["X_train"].shape[0]
        te = stead_split_indices(n)[2]
    else:
        path = find_one(INSTANCE_FILE)
        with h5py.File(path, "r") as f:
            te = np.where(f["split_id"][:] == 2)[0]
    pick = rng.choice(te, size=min(SYN_POOL_ROWS, len(te)), replace=False)
    t1 = time.time(); X, Y = load_rows(path, pick)
    print("%s: %d test rows read from %s in %.0f s" % (source, len(X), path.name, time.time() - t1))
    return X, Y

def quake_marks(y):
    """(onset, start, end) samples of a clean earthquake window: 5 % of peak, 1 % of peak, 95 % of energy."""
    a = np.abs(y).max(1); pk = a.max()
    e = np.cumsum((y ** 2).sum(1))
    return int(np.argmax(a >= 0.05 * pk)), int(np.argmax(a >= 0.01 * pk)), int(np.searchsorted(e, 0.95 * e[-1]))

def build_streams(source):
    f = RESULTS / "synth" / ("%s_streams.npz" % source)
    if RESUME and f.exists():
        z = np.load(f, allow_pickle=True)
        n = int(z["n_streams"])
        return [dict(x=z["x%d" % i], truth=z["t%d" % i]) for i in range(n)], pd.read_csv(RESULTS / "synth" / ("%s_events.csv" % source))
    rng = np.random.default_rng(SYN_SEED + SYN_SOURCES.index(source))
    X, Y = test_pool(source, rng)
    noise = X - Y
    nstd = noise.std(1)
    noise_ok = np.where((nstd > 0).all(1))[0]
    marks = [quake_marks(y) if (y.std(0) > 0).all() else (-1, -1, -1) for y in Y]
    lo, hi = int(SYN_ONSET_RANGE_S[0] * FS), int(SYN_ONSET_RANGE_S[1] * FS)
    eq_ok = np.array([i for i, m in enumerate(marks) if lo <= m[0] <= hi])
    print("%s: %d usable earthquakes, %d usable noise windows" % (source, len(eq_ok), len(noise_ok)))
    snrs = np.tile(SYN_SNR, int(np.ceil(SYN_EVENTS / len(SYN_SNR))))[:SYN_EVENTS]
    rng.shuffle(snrs)
    L, cf = int(SYN_STREAM_H * 3600 * FS), int(SYN_CROSSFADE_S * FS)
    fin, fout = np.sin(np.linspace(0, np.pi / 2, cf))[:, None], np.cos(np.linspace(0, np.pi / 2, cf))[:, None]
    streams, events, k_ev = [], [], 0
    while k_ev < SYN_EVENTS:
        out = noise[rng.choice(noise_ok)].copy(); out /= out.std(0)
        pieces = [out]
        total = WIN
        while total < L:
            j = rng.choice(noise_ok); p = noise[j] / noise[j].std(0)
            pieces.append(p); total += WIN - cf
        x = np.zeros((total, 3)); x[:WIN] = pieces[0]; pos = WIN
        for p in pieces[1:]:
            x[pos - cf:pos] = x[pos - cf:pos] * fout + p[:cf] * fin
            x[pos:pos + WIN - cf] = p[cf:]
            pos += WIN - cf
        x = x[:L]
        truth = np.zeros_like(x)
        t = (LTA_S + 60.0) * FS + rng.uniform(0, SYN_GAP_S[0]) * FS
        while k_ev < SYN_EVENTS and t + 70 * FS < L:
            j = rng.choice(eq_ok); on, st0, en = marks[j]
            q = Y[j] / Y[j].std(0) * snrs[k_ev]
            p0 = int(t) - on
            truth[p0:p0 + WIN] += q
            events.append(dict(stream=len(streams), onset=int(t), start=p0 + st0, end=p0 + en, snr=float(snrs[k_ev]), row=int(j)))
            k_ev += 1
            t += rng.uniform(*SYN_GAP_S) * FS
        streams.append(dict(x=x + truth, truth=truth))
    ev = pd.DataFrame(events)
    (RESULTS / "synth").mkdir(parents=True, exist_ok=True)
    np.savez_compressed(f, n_streams=len(streams), **{("x%d" % i): s["x"].astype(np.float32) for i, s in enumerate(streams)},
                        **{("t%d" % i): s["truth"].astype(np.float32) for i, s in enumerate(streams)})
    ev.to_csv(RESULTS / "synth" / ("%s_events.csv" % source), index=False)
    print("%s: %d streams x %.1f h, %d events" % (source, len(streams), SYN_STREAM_H, len(ev)))
    return streams, ev

def synth_one(source, mid, pred, streams, ev):
    t1 = time.time()
    lo_d, hi_d = int(SYN_DETECT[0] * FS), int(SYN_DETECT[1] * FS)
    rows = {thr: dict(det=np.zeros(len(ev), bool), terr=np.full(len(ev), np.nan), false=0) for thr in TRIG_ON_SWEEP}
    ev_val = np.full(len(ev), np.nan); hours = 0.0
    for si, S in enumerate(streams):
        x = S["x"].astype(np.float64)
        eq = baseline_or_model(mid, x, pred)[0]
        valid = np.ones(len(x), bool); valid[: int(LTA_S * FS)] = False
        hours += valid.sum() / FS / 3600
        trig = sweep_triggers(eq, valid)
        e = ev[ev.stream == si]
        for i, r in e.iterrows():
            sl = slice(int(r.start), int(r.end) + 1)
            ev_val[i] = ev_rows(S["truth"][sl].astype(np.float64), eq[sl])
        for thr, t in trig.items():
            t = np.sort(t)
            a, b = np.searchsorted(t, e.onset.values - lo_d), np.searchsorted(t, e.onset.values + hi_d, side="right")
            for k, (i, j) in enumerate(zip(a, b)):
                if j > i:
                    rows[thr]["det"][e.index[k]] = True
                    rows[thr]["terr"][e.index[k]] = (t[i] - e.onset.values[k]) / FS
            belongs = np.zeros(len(t), bool)
            for i, j in zip(np.searchsorted(t, e.start.values - int(SYN_SPAN_PAD[0] * FS)), np.searchsorted(t, e.end.values + int(SYN_SPAN_PAD[1] * FS), side="right")):
                belongs[i:j] = True
            rows[thr]["false"] += int((~belongs).sum())
    out = []
    for thr, r in rows.items():
        for snr in [None] + list(SYN_SNR):
            m = np.ones(len(ev), bool) if snr is None else np.isclose(ev.snr.values, snr)
            out.append(dict(model_id=mid, source=source, thr_on=thr, snr="all" if snr is None else snr, n=int(m.sum()),
                            det=float(r["det"][m].mean()), false_per_h=r["false"] / hours,
                            t_err_median=float(np.nanmedian(r["terr"][m])) if r["det"][m].any() else np.nan,
                            t_err_abs_p90=float(np.nanpercentile(np.abs(r["terr"][m]), 90)) if r["det"][m].any() else np.nan,
                            ev_median=float(np.nanmedian(ev_val[m]))))
    df = pd.DataFrame(out)
    df.to_csv(RESULTS / "synth" / source / ("%s.csv" % mid), index=False)
    pd.DataFrame(dict(ev=ev_val, det6=rows[TRIG_ON]["det"].astype(int), terr6=rows[TRIG_ON]["terr"])).to_csv(RESULTS / "synth" / source / ("%s_events.csv" % mid), index=False)
    a = df[(df.thr_on == TRIG_ON) & (df.snr == "all")].iloc[0]
    print("  %-26s det %.3f  false/h %6.2f  t_err %5.2f s  EV %.3f  (%.0f s)" % (mid, a.det, a.false_per_h, a.t_err_median, a.ev_median, time.time() - t1))
    return df

if RUN_SYNTH:
    for source in SYN_SOURCES:
        (RESULTS / "synth" / source).mkdir(parents=True, exist_ok=True)
        try:
            streams, ev = build_streams(source)
        except FileNotFoundError as e:
            print("skip %s: %s" % (source, e)); continue
        print("== %s: %d streams, %d events" % (source, len(streams), len(ev)))
        for mid in list(BASELINES) + list(MODELS):
            if RESUME and (RESULTS / "synth" / source / ("%s.csv" % mid)).exists():
                print("  %-26s done (RESUME)" % mid); continue
            pred = None if mid in BASELINES else model_predictor(load_model(MODELS[mid]["path"]))
            synth_one(source, mid, pred, streams, ev)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        log("Part 4: %s done, %d events" % (source, len(ev)))
else:
    print("Part 4 skipped")

STEAD: 4000 test rows read from training_datasets_all_snr_40_unshuffled.hdf5 in 24 s
STEAD: 1646 usable earthquakes, 3968 usable noise windows
STEAD: 11 streams x 6.0 h, 1400 events
== STEAD: 11 streams, 1400 events
  raw                        det 0.381  false/h   0.70  t_err  0.40 s  EV 0.705  (1 s)
  bandpass                   det 0.419  false/h   2.60  t_err  0.30 s  EV 0.906  (1 s)
  stead_None_seed99          det 0.500  false/h   6.96  t_err  0.20 s  EV 0.954  (4 s)
  stead_Linear_seed99        det 0.504  false/h   7.16  t_err  0.20 s  EV 0.951  (3 s)
  stead_LSTM_seed99          det 0.455  false/h   4.41  t_err  0.20 s  EV 0.968  (5 s)
  stead_attention_seed99     det 0.457  false/h   4.76  t_err  0.20 s  EV 0.968  (4 s)
  stead_Transformer_seed99   det 0.486  false/h   6.96  t_err  0.20 s  EV 0.957  (8 s)
  instance_None_seed99       det 0.491  false/h   8.86  t_err  0.20 s  EV 0.896  (3 s)
  instance_Linear_seed99     det 0.491  false/h   8.72  t_err  0.20 s  EV 0.901  (4 s)
 

## Part 5 : Tables and figures

In [ ]:
if RUN_FIGURES or RUN_PAPER_FIGS:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch

    C_PAPER, C_NEW, C_3, C_4 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
    SERIES = [C_PAPER, C_NEW, C_3, C_4]
    INK, INK2, GRID, SURF, MUTED = "#0b0b0b", "#52514e", "#e5e4e0", "#fcfcfb", "#9a9a94"

    plt.rcParams.update({ "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
        "font.size": 9, "axes.titlesize": 9.5, "axes.labelsize": 9,
        "axes.edgecolor": INK2, "axes.linewidth": 0.8,
        "axes.spines.top": False, "axes.spines.right": False,
        "text.color": INK, "axes.labelcolor": INK,
        "xtick.color": INK2, "ytick.color": INK2,
        "xtick.labelsize": 8, "ytick.labelsize": 8,
        "grid.color": GRID, "grid.linewidth": 0.7,
        "legend.frameon": False, "legend.fontsize": 8,
        "lines.linewidth": 1.8, "lines.markersize": 5,
        "savefig.dpi": 300, "savefig.bbox": "tight",
        "pdf.fonttype": 42, "ps.fonttype": 42,})
    ORDER = list(PAPER_BOTTLENECKS) + ["attention_PE", "hybrid", "attention_LSTM", "Transformer_fixed"]

    def bcolor(kind):
        """Same mapping as notebook 07: paper bottleneck blue, new bottleneck orange, authors' pretrained green, baseline grey."""
        return {"paper": C_PAPER, "new": C_NEW, "pretrained": C_3}.get(kind, MUTED)

    def border(ax, grid_axis="y"):
        ax.grid(True, axis=grid_axis, zorder=0)
        ax.set_axisbelow(True)

    def save_fig(fig, name):
        for ext in ("pdf", "png"):
            fig.savefig(FIGURES / ("%s.%s" % (name, ext)))
        plt.close(fig)
        print("  figure: %s (.pdf, .png)" % name)

    def order_key(b):
        return ORDER.index(b) if b in ORDER else len(ORDER)

    def meta(mid):
        """group / kind / bottleneck of a model id, also for ids no longer in MODELS."""
        if mid in BASELINES:
            return dict(group=mid, kind="baseline", bottleneck=mid, trained_on="")
        if mid in MODELS:
            return MODELS[mid]
        if mid.startswith("pretrained_"):
            return dict(group="pretrained", kind="pretrained", bottleneck=mid[len("pretrained_"):], trained_on="AUTHORS")
        on, rest = mid.split("_", 1)
        b = re.sub(r"_seed\d+$", "", rest)
        new = b in NEW_BOTTLENECKS
        return dict(group="variant" if new else on, kind="new" if new else "paper", bottleneck=b, trained_on=on.upper())

    # one panel per model group, the colour is the group's kind colour (07), shaded per bottleneck, with one marker per bottleneck
    PANELS = (("stead", "(a) paper bottlenecks, trained on STEAD"), ("instance", "(b) paper bottlenecks, trained on INSTANCE"),
              ("variant", "(c) new bottlenecks, trained on STEAD"), ("pretrained", "(d) authors' pretrained"))
    MARKERS = {"None": "o", "Linear": "s", "LSTM": "^", "attention": "D", "Transformer": "v",
               "attention_PE": "P", "hybrid": "*", "attention_LSTM": "X", "Transformer_fixed": "h"}

    def shade(hex_color, k, n):
        """k-th of n shades of one colour, from full colour to 55 % mixed with the background."""
        import matplotlib.colors as mc
        c, s = np.array(mc.to_rgb(hex_color)), np.array(mc.to_rgb(SURF))
        f = 0.55 * k / max(n - 1, 1)
        return tuple((1 - f) * c + f * s)

    def style(mid, members=None):
        m = meta(mid)
        if mid == "raw":
            return dict(color=INK2, ls="--", lw=1.6, marker=None, label="raw (no separation)", zorder=5)
        if mid == "bandpass":
            return dict(color=MUTED, ls=":", lw=1.8, marker=None, label="band-pass %g-%g Hz" % BANDPASS_HZ, zorder=5)
        members = members or [mid]
        k = sorted(members, key=lambda x: order_key(meta(x)["bottleneck"])).index(mid)
        return dict(color=shade(bcolor(m["kind"]), k, len(members)), ls="-", lw=1.5, marker=MARKERS.get(m["bottleneck"], "o"),
                    label=m["bottleneck"], zorder=3)

    def panel_members(ids, group):
        return sorted([m for m in ids if meta(m)["group"] == group], key=lambda x: order_key(meta(x)["bottleneck"]))

    def panel_figure(ids, plot_one, xlabel, ylabel, suptitle, xlog=False, xsymlog=None):
        """2 x 2 panels (STEAD / INSTANCE / new / pretrained); plot_one(ax, mid, style) draws one curve; raw is drawn in every panel."""
        groups = [(g, t) for g, t in PANELS if panel_members(ids, g)]
        if not groups:
            return None
        ncol = 2 if len(groups) > 1 else 1
        nrow = int(np.ceil(len(groups) / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(4.8 * ncol, 3.4 * nrow), sharex=True, sharey=True, squeeze=False)
        for ax in axes.ravel()[len(groups):]:
            ax.axis("off")
        for ax, (g, title) in zip(axes.ravel(), groups):
            mem = panel_members(ids, g)
            for b in BASELINES:
                if b in ids:
                    plot_one(ax, b, style(b))
            for mid in mem:
                plot_one(ax, mid, style(mid, mem))
            if xlog:
                ax.set_xscale("log")
            if xsymlog is not None:
                ax.set_xscale("symlog", linthresh=xsymlog)
            border(ax, grid_axis="both")
            ax.set_title(title, loc="left")
            ax.legend(loc="best", fontsize=7)
        for ax in axes[-1]:
            ax.set_xlabel(xlabel)
        for ax in axes[:, 0]:
            ax.set_ylabel(ylabel)
        fig.suptitle(suptitle, x=0.01, ha="left", fontsize=9.5)
        fig.tight_layout()
        return fig

    def bar_pair(d, cols, xlabels, title, name, fmts=("%.3f", "%.2f"), xlog=(False, True)):
        d = d.sort_values(cols[0]).reset_index(drop=True)
        y = np.arange(len(d))
        fig, axes = plt.subplots(1, 2, figsize=(10.2, 0.30 * len(d) + 1.7), sharey=True)
        for ax, col, xl, fm, lg in zip(axes, cols, xlabels, fmts, xlog):
            v = d[col].astype(float).fillna(0).values
            ax.barh(y, v, height=0.62, color=[bcolor(meta(m)["kind"]) for m in d.model_id], zorder=3)
            for yy, vv in zip(y, v):
                ax.annotate(fm % vv, (vv, yy), xytext=(5, 0), textcoords="offset points", va="center", fontsize=7.5, color=INK)
            if lg:
                ax.set_xscale("symlog", linthresh=1.0)
            ax.set_xlim(0, (v.max() if len(v) else 1) * (3.0 if lg else 1.18) + 1e-9)
            ax.set_yticks(y)
            ax.set_yticklabels(d.model_id, fontsize=7.5)
            ax.set_xlabel(xl)
            border(ax, grid_axis="x")
        kinds = {meta(m)["kind"] for m in d.model_id}
        lab = {"paper": "paper bottleneck", "new": "new bottleneck", "pretrained": "authors' pretrained", "baseline": "baseline (raw, band-pass)"}
        fig.legend(handles=[Patch(facecolor=bcolor(k), label=lab[k]) for k in ("paper", "new", "pretrained", "baseline") if k in kinds],
                   loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.04))
        fig.suptitle(title, x=0.01, ha="left", fontsize=9.5)
        fig.tight_layout()
        save_fig(fig, name)

    def read_csv_keep(p):
        return pd.read_csv(p, keep_default_na=False, na_values=[""])

    print("figure style ready (notebook 07); output ->", FIGURES)
else:
    print("Part 5 skipped")

figure style ready (notebook 07); output -> /kaggle/working/figures_08


In [ ]:
# Part 5a : Real data
if RUN_FIGURES:
    ids_all = list(BASELINES) + list(MODELS)
    summ, sweep, bymag = [], [], []
    for tag_dir in (sorted((RESULTS / "real").glob("*")) if (RESULTS / "real").exists() else []):
        tag = tag_dir.name
        R = {mid: json.load(open(tag_dir / ("%s.json" % mid))) for mid in ids_all if (tag_dir / ("%s.json" % mid)).exists()}
        if "raw" not in R:
            print("  %s: no raw result, skipped" % tag)
            continue
        ev = pd.read_csv(RESULTS / ("catalogue_%s.csv" % tag))
        u = ev[ev.usable].reset_index(drop=True)
        for mid, r in R.items():
            s6 = r["sweep"][str(TRIG_ON)]
            summ.append(dict(station=tag, model_id=mid, kind=meta(mid)["kind"], group=meta(mid)["group"], n_trig=s6["n_trig"], recall=s6["recall"],
                             recall_m2=s6["recall_m2"], uncat_per_h=s6["uncat_per_h"], snr_gain_db=r["snr_gain_db_median"],
                             quiet_leak=r["quiet_leak_median"], residual=r["residual"], boundary_frac=r["boundary_frac"], n_events=len(u)))
            for thr, s in r["sweep"].items():
                sweep.append(dict(station=tag, model_id=mid, thr_on=float(thr), n_trig=s["n_trig"], recall=s["recall"],
                                  recall_m2=s["recall_m2"], uncat_per_h=s["uncat_per_h"]))
            for lo, hi in zip(MAG_BINS[:-1], MAG_BINS[1:]):
                k = "%g-%g" % (lo, hi)
                bymag.append(dict(station=tag, model_id=mid, mag_lo=lo, mag_hi=hi, recall=s6["recall_by_mag"].get(k),
                                  n=int(((u.mag >= lo) & (u.mag < hi)).sum())))
        SU = pd.DataFrame([x for x in summ if x["station"] == tag])
        SW = pd.DataFrame([x for x in sweep if x["station"] == tag])
        BM = pd.DataFrame([x for x in bymag if x["station"] == tag])
        ids = list(R)

        # (1) summary bars at the paper threshold
        bar_pair(SU, ("recall", "uncat_per_h"), ("share of %d catalogue events detected" % len(u), "uncatalogued triggers per hour"),
                 "%s, %s + %d days, STA/LTA %g/%g s, trigger_on = %g, rows ordered by detection" % (tag, START, N_DAYS, STA_S, LTA_S, TRIG_ON),
                 "fig_real_summary_%s" % tag)

        # (2) threshold sweep: detection vs uncatalogued triggers, one panel per group
        def _sweep(ax, mid, st):
            d = SW[SW.model_id == mid].sort_values("thr_on")
            ax.plot(d.uncat_per_h, d.recall, markersize=4, **st)
            p6 = d[d.thr_on == TRIG_ON]
            ax.plot(p6.uncat_per_h, p6.recall, "o", color=st["color"], ms=8, markeredgecolor=INK, markeredgewidth=0.8, zorder=6)
        fig = panel_figure(ids, _sweep, "uncatalogued triggers per hour (upper bound on false alarms)", "share of catalogue events detected",
                           "%s: trigger_on swept over %s, ringed dot = %g; up and to the left is better" % (tag, ", ".join("%g" % t for t in TRIG_ON_SWEEP), TRIG_ON),
                           xlog=True)
        if fig is not None:
            save_fig(fig, "fig_real_sweep_%s" % tag)

        # (3) detection per magnitude bin
        xs = np.arange(len(MAG_BINS) - 1)
        xt = ["M<%g" % MAG_BINS[1]] + ["M%g-%g" % (a, b) for a, b in zip(MAG_BINS[1:-2], MAG_BINS[2:-1])] + ["M>=%g" % MAG_BINS[-2]]
        nb_ = BM[BM.model_id == "raw"].n.values
        def _mag(ax, mid, st):
            d = BM[BM.model_id == mid]
            ax.plot(xs, d.recall.astype(float).values, markersize=4, **st)
            ax.set_xticks(xs)
            ax.set_xticklabels(["%s\n(n=%d)" % (a, n) for a, n in zip(xt, nb_)], fontsize=7)
        fig = panel_figure(ids, _mag, "catalogue magnitude", "share detected at trigger_on = %g" % TRIG_ON, "%s: catalogue events detected per magnitude bin" % tag)
        if fig is not None:
            save_fig(fig, "fig_real_recall_mag_%s" % tag)

        # (4) waveform examples: grey raw, kind colour separated earthquake, yellow TauP P (solid) and S (dashed)
        raw_s = np.load(tag_dir / "raw_snippets.npz")
        keys = list(raw_s.keys())
        show = list(FIG_MODELS_EXAMPLES) if FIG_MODELS_EXAMPLES else [m for m in ids if meta(m)["bottleneck"] == "LSTM"]
        show = (["bandpass"] if "bandpass" in R else []) + [m for m in show if m in R and m not in BASELINES]
        if keys and show:
            comp = 2
            fig, axes = plt.subplots(len(show) + 1, len(keys), figsize=(3.1 * len(keys), 1.05 * (len(show) + 1) + 1.0), sharex=True, squeeze=False)
            tt = np.arange(-300, 900) / FS
            for j, k in enumerate(keys):
                e = u.iloc[int(np.argmin(np.abs(u.tP.values - float(k))))]
                r0 = raw_s[k][:, comp] - raw_s[k][:, comp].mean()
                for i, mid in enumerate(["raw"] + show):
                    ax = axes[i, j]
                    ax.plot(tt, r0 / (np.abs(r0).max() + 1e-12), color=MUTED, lw=0.7, zorder=2)
                    if mid != "raw":
                        s = np.load(tag_dir / ("%s_snippets.npz" % mid))[k][:, comp]
                        s = s - s.mean()
                        ax.plot(tt, s / (np.abs(s).max() + 1e-12), color=INK2 if mid == "bandpass" else bcolor(meta(mid)["kind"]), lw=0.8, zorder=3)
                    ax.axvline(0, color=C_4, lw=1.0, zorder=4)
                    ax.axvline(e.tS - e.tP, color=C_4, lw=1.0, ls="--", zorder=4)
                    ax.set_ylim(-1.05, 1.05)
                    ax.set_yticks([])
                    if j == 0:
                        ax.set_ylabel("raw" if mid == "raw" else mid.replace("_seed99", ""), rotation=0, ha="right", va="center", fontsize=7.5)
                axes[0, j].set_title("M%.1f, %.0f km, %s" % (e.mag, e.dist_deg * 111.2, e.origin[:16].replace("T", " ")), loc="left", fontsize=8)
                axes[-1, j].set_xlabel("s after TauP P")
            lab = {"paper": "separated, paper bottleneck", "new": "separated, new bottleneck", "pretrained": "separated, authors' pretrained"}
            h = [Line2D([], [], color=MUTED, lw=1.2, label="raw")] + ([Line2D([], [], color=INK2, lw=1.2, label="band-pass %g-%g Hz" % BANDPASS_HZ)] if "bandpass" in show else []) + [Line2D([], [], color=bcolor(k), lw=1.2, label=lab[k]) for k in lab if k in {meta(m)["kind"] for m in show if m not in BASELINES}]
            h += [Line2D([], [], color=C_4, lw=1.2, label="TauP P"), Line2D([], [], color=C_4, lw=1.2, ls="--", label="TauP S")]
            fig.legend(handles=h, loc="lower center", ncol=len(h), bbox_to_anchor=(0.5, -0.03))
            fig.suptitle("%s, channel %s, every trace scaled to its own peak (amplitudes: see the summary table)" % (tag, (R["raw"].get("channels") or ["", "", "Z"])[comp]),
                         x=0.01, ha="left", fontsize=9.5)
            fig.tight_layout()
            save_fig(fig, "fig_real_examples_%s" % tag)

        # (5) triggers per hour against the catalogue (the paper's step7a figure, over the whole period)
        sel = [b for b in BASELINES if b in R] + [m for m in FIG_MODELS_HOURLY if m in R]
        nh = len(R["raw"]["hourly"])
        fig, ax = plt.subplots(figsize=(10.2, 3.2))
        for i, mid in enumerate(sel):
            h = np.array(R[mid]["hourly"])
            if mid in BASELINES:
                st = style(mid)
                ax.step(np.arange(nh) / 24, h, where="post", color=st["color"], ls=st["ls"], lw=1.0, label=st["label"], zorder=2)
            else:
                ax.step(np.arange(nh) / 24, h, where="post", color=bcolor(meta(mid)["kind"]), lw=1.0,
                        ls=("-", "--", "-.", ":")[i % 4], label=mid.replace("_seed99", ""), zorder=3)
        cat_h = np.histogram(ev.tP.values, bins=np.arange(nh + 1) * 3600)[0]
        ax.plot((np.arange(nh) + .5) / 24, np.where(cat_h > 0, cat_h, np.nan), "o", color=C_4, ms=3, label="catalogue events (all M)", zorder=4)
        ax.set_yscale("symlog", linthresh=1)
        ax.set_xlabel("days since %s" % START)
        ax.set_ylabel("per hour")
        border(ax)
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=len(sel) + 1, fontsize=7.5)
        ax.set_title("%s: STA/LTA triggers per hour at trigger_on = %g" % (tag, TRIG_ON), loc="left")
        fig.tight_layout()
        save_fig(fig, "fig_real_hourly_%s" % tag)

    if summ:
        pd.DataFrame(summ).to_csv(RESULTS / "table_real_summary.csv", index=False)
        pd.DataFrame(sweep).to_csv(RESULTS / "table_real_sweep.csv", index=False)
        pd.DataFrame(bymag).to_csv(RESULTS / "table_real_recall_by_mag.csv", index=False)
        with pd.option_context("display.width", 220, "display.max_columns", 20):
            print(pd.DataFrame(summ).drop(columns=["group"]).round(3).to_string(index=False))
    else:
        print("  no real-data results yet (run Part 3)")

  figure: fig_real_summary_HV.HAT (.pdf, .png)
  figure: fig_real_sweep_HV.HAT (.pdf, .png)
  figure: fig_real_recall_mag_HV.HAT (.pdf, .png)
  figure: fig_real_examples_HV.HAT (.pdf, .png)
  figure: fig_real_hourly_HV.HAT (.pdf, .png)
  figure: fig_real_summary_IU.POHA (.pdf, .png)
  figure: fig_real_sweep_IU.POHA (.pdf, .png)
  figure: fig_real_recall_mag_IU.POHA (.pdf, .png)
  figure: fig_real_examples_IU.POHA (.pdf, .png)
  figure: fig_real_hourly_IU.POHA (.pdf, .png)
station                       model_id       kind  n_trig  recall  recall_m2  uncat_per_h  snr_gain_db  quiet_leak  residual  boundary_frac  n_events
 HV.HAT                            raw   baseline     190   0.039      0.070        0.503        0.000       1.000     0.000          0.042       483
 HV.HAT                       bandpass   baseline     788   0.352      0.480        1.834        3.297       0.011     0.000          0.046       483
 HV.HAT              stead_None_seed99      paper   12530   0.698      0.

In [13]:
# Part 5b: synthetic streams
if RUN_FIGURES:
    allsyn = []
    LOW_SNR = tuple(s for s in SYN_SNR if s <= 1.0)
    for source in SYN_SOURCES:
        d = RESULTS / "synth" / source
        S = {mid: read_csv_keep(d / ("%s.csv" % mid)) for mid in list(BASELINES) + list(MODELS) if (d / ("%s.csv" % mid)).exists()}
        if "raw" not in S:
            print("  %s: no synthetic results yet (run Part 4)" % source)
            continue
        for df in S.values():
            df["snr"] = df.snr.astype(str)
        allsyn += list(S.values())
        ids = list(S)

        def at_snr(df, snr_set):
            x = df[df.snr.isin([str(float(s)) for s in snr_set])]
            out = [dict(thr_on=t, det=np.average(q.det, weights=q.n), fph=q.false_per_h.iloc[0]) for t, q in x.groupby("thr_on")]
            return pd.DataFrame(out).sort_values("thr_on")

        # (1) summary bars: detection at SNR <= 1 and false alarms per hour, paper threshold
        rows = []
        for mid, df in S.items():
            g = at_snr(df, LOW_SNR)
            g6 = g[g.thr_on == TRIG_ON].iloc[0]
            a = df[(df.thr_on == TRIG_ON) & (df.snr == "all")].iloc[0]
            rows.append(dict(model_id=mid, det_low=g6.det, det_all=a.det, false_per_h=a.false_per_h))
        bar_pair(pd.DataFrame(rows), ("det_low", "false_per_h"), ("share detected, inserted SNR <= 1", "false alarms per hour"),
                 "%s test split, synthetic streams, trigger_on = %g, rows ordered by detection at SNR <= 1" % (source, TRIG_ON),
                 "fig_synth_summary_%s" % source)

        # (2) detection and (3) EV vs SNR
        for kind, col, ylab in (("det_vs_snr", "det", "share detected"), ("ev_vs_snr", "ev_median", "median EV over the event span")):
            def _snr(ax, mid, st, col=col):
                x = S[mid][(S[mid].thr_on == TRIG_ON) & (S[mid].snr != "all")].copy()
                x["s"] = x.snr.astype(float)
                x = x.sort_values("s")
                ax.plot(x.s, x[col], markersize=4, **st)
                if col == "ev_median":
                    ax.axhline(0.0, color=MUTED, lw=0.8, zorder=1)
            fig = panel_figure(ids, _snr, "inserted SNR (std ratio, as the mixtures)", ylab,
                               "%s test split, synthetic streams, trigger_on = %g" % (source, TRIG_ON), xlog=True)
            if fig is not None:
                save_fig(fig, "fig_synth_%s_%s" % (kind, source))

        # (4) ROC over the threshold sweep, SNR <= 1
        def _roc(ax, mid, st):
            g = at_snr(S[mid], LOW_SNR)
            ax.plot(g.fph, g.det, markersize=4, **st)
            p6 = g[g.thr_on == TRIG_ON]
            ax.plot(p6.fph, p6.det, "o", color=st["color"], ms=8, markeredgecolor=INK, markeredgewidth=0.8, zorder=6)
        fig = panel_figure(ids, _roc, "false alarms per hour", "share detected, inserted SNR <= 1",
                           "%s: trigger_on swept over %s, ringed dot = %g; up and to the left is better" % (source, ", ".join("%g" % t for t in TRIG_ON_SWEEP), TRIG_ON),
                           xsymlog=0.1)
        if fig is not None:
            save_fig(fig, "fig_synth_roc_%s" % source)

    if allsyn:
        T = pd.concat(allsyn)
        T.to_csv(RESULTS / "table_synth.csv", index=False)
        with pd.option_context("display.width", 220):
            print(T[(T.thr_on == TRIG_ON) & (T.snr == "all")].round(3).to_string(index=False))

    print("\nfigures written to", FIGURES)
    for p in sorted(FIGURES.glob("*.pdf")):
        print("   ", p.name)
    log("Part 5: %d figures" % len(list(FIGURES.glob("*.pdf"))))

  figure: fig_synth_summary_STEAD (.pdf, .png)
  figure: fig_synth_det_vs_snr_STEAD (.pdf, .png)
  figure: fig_synth_ev_vs_snr_STEAD (.pdf, .png)
  figure: fig_synth_roc_STEAD (.pdf, .png)
  figure: fig_synth_summary_INSTANCE (.pdf, .png)
  figure: fig_synth_det_vs_snr_INSTANCE (.pdf, .png)
  figure: fig_synth_ev_vs_snr_INSTANCE (.pdf, .png)
  figure: fig_synth_roc_INSTANCE (.pdf, .png)
                      model_id   source  thr_on snr    n   det  false_per_h  t_err_median  t_err_abs_p90  ev_median
                           raw    STEAD     6.0 all 1400 0.381        0.699           0.4           3.10      0.705
                      bandpass    STEAD     6.0 all 1400 0.419        2.598           0.3           3.20      0.906
             stead_None_seed99    STEAD     6.0 all 1400 0.500        6.959           0.2           2.90      0.954
           stead_Linear_seed99    STEAD     6.0 all 1400 0.504        7.156           0.2           2.85      0.951
             stead_LSTM_seed99

## Part 6 : STA/LTA figures in the layout of the paper

In [ ]:
# Part 6: STA/LTA Figures
if RUN_PAPER_FIGS:
    import matplotlib.dates as mdates
    from obspy import UTCDateTime
    if not STREAMS:
        prepare_streams()
    PF = FIGURES / "paper_stalta"
    PF.mkdir(parents=True, exist_ok=True)
    COMP_COL = (C_PAPER, C_NEW, C_3)
    hop = int(round(WIN * (1 - OVERLAP)))

    def run_detector(tag, mid):
        S = STREAMS[tag]; w = S["w"]
        pred = None if mid in BASELINES else model_predictor(load_model(MODELS[mid]["path"]))
        gstats = (w[S["good"]].mean(0), w[S["good"]].std(0) + 1e-12)
        x = baseline_or_model(mid, w, pred, gstats=gstats)[0]
        cf = cft3(x)
        t = coincidence(cf, TRIG_ON, min(TRIG_OFF, TRIG_ON - 0.5))[:, 0]
        if len(t):
            t = t[S["valid"][t]]
        return x, cf, t

    def detected(u, t):
        ts = np.sort(t) / FS
        return np.searchsorted(ts, u.tP.values + MATCH_POST, side="right") > np.searchsorted(ts, u.tP.values - MATCH_PRE)

    def pick_examples(ev, t_raw, t_bp, t_mod):
        u = ev[ev.usable].copy()
        u["raw"], u["bandpass"], u["model"] = detected(u, t_raw), detected(u, t_bp), detected(u, t_mod)
        out = []
        a = u[u.raw & u.model]
        if len(a):
            out.append(("detected_by_both", a.loc[a.mag.idxmax()]))
        b = u[~u.raw & u.model].sort_values("mag", ascending=False)
        for k in sorted({0, len(b) // 2}) if len(b) else []:
            out.append(("separated_only", b.iloc[k]))
        c = u[~u.model].sort_values("mag", ascending=False)
        if len(c):
            out.append(("missed_by_model", c.iloc[0]))
        ts = np.sort(t_mod) / FS
        far = [t for t in ts if np.all((t < ev.tP.values - 120) | (t > ev.tS.values + 120))]
        if far:
            out.append(("uncatalogued_trigger", pd.Series(dict(tP=far[len(far) // 2], tS=np.nan, mag=np.nan, dist_deg=np.nan, origin="", raw=False, bandpass=False, model=True))))
        for t in (PAPER_TIMES or []):
            tt = UTCDateTime(t) - UTCDateTime(STREAMS_START)
            out.append(("user_time", pd.Series(dict(tP=tt, tS=np.nan, mag=np.nan, dist_deg=np.nan, origin=str(t), raw=False, bandpass=False, model=False))))
        return out

    def paper_zoom(tag, mid, raw, cf_raw, t_raw, sep, cf_sep, t_sep, centre, case, row, fname):
        S = STREAMS[tag]; ev = S["events"]
        i0 = max(int((centre - PAPER_WINDOW_S[0]) * FS), 0)
        i1 = min(int((centre + PAPER_WINDOW_S[1]) * FS), len(raw))
        t0 = UTCDateTime(S["meta"]["start"])
        tax = np.datetime64(t0.datetime) + (np.arange(i0, i1) * (1e6 / FS)).astype("timedelta64[us]")
        def tdt(samples_or_s, unit="samples"):
            s = np.asarray(samples_or_s, float) / (FS if unit == "samples" else 1.0)
            return np.datetime64(t0.datetime) + (s * 1e6).astype("timedelta64[us]")
        chans = S["meta"]["channels"]
        lab = "raw" if mid == "raw" else mid.replace("_seed99", "")
        cols = (("(a) raw waveform", raw, cf_raw, t_raw), ("(b) separated earthquake, %s" % lab, sep, cf_sep, t_sep))
        evw = ev[(ev.tP >= i0 / FS) & (ev.tP < i1 / FS)]
        evs = ev[(ev.tS >= i0 / FS) & (ev.tS < i1 / FS)]
        fig, ax = plt.subplots(4, 2, sharex=True, figsize=(10.2, 8.4))
        cmax = 0.0
        for j, (name, x, cf, tr) in enumerate(cols):
            a = ax[0, j]
            for c in range(3):
                a.plot(tax, cf[c][i0:i1], color=COMP_COL[c], lw=1.1, label=chans[c], zorder=3)
                cmax = max(cmax, float(np.nanmax(cf[c][i0:i1])))
            a.axhline(TRIG_ON, color=INK, lw=1.0, label="trigger on (%g)" % TRIG_ON, zorder=4)
            a.axhline(TRIG_OFF, color=INK, lw=1.0, ls="--", label="trigger off (%g)" % TRIG_OFF, zorder=4)
            trw = tr[(tr >= i0) & (tr < i1)]
            a.set_title("%s, %d trigger%s" % (name, len(trw), "" if len(trw) == 1 else "s"), loc="left")
            border(a)
            for c in range(3):
                a = ax[c + 1, j]
                seg = x[i0:i1, c] - x[i0:i1, c].mean()
                a.plot(tax, seg, color=COMP_COL[c], lw=0.7, zorder=3)
                for k in range(-(-i0 // hop) * hop, i1, hop):
                    a.axvline(tdt(k), color=MUTED, lw=0.4, zorder=1)
                a.plot(tdt(trw), np.zeros(len(trw)), "x", color=INK, ms=10, mew=1.8, zorder=10)
                a.plot(tdt(evw.tP.values, "s"), np.zeros(len(evw)), "o", ms=5, color=C_4, mec=INK, mew=1.0, zorder=9)
                a.plot(tdt(evs.tS.values, "s"), np.zeros(len(evs)), "o", ms=9, color=C_4, mec=INK, mew=1.0, zorder=8, alpha=0.9)
                border(a)
                if j == 0:
                    a.set_ylabel("%s component" % chans[c])
        for c in range(3):
            y = 1.1 * max(np.abs(cols[0][1][i0:i1, c] - cols[0][1][i0:i1, c].mean()).max(), np.abs(cols[1][1][i0:i1, c] - cols[1][1][i0:i1, c].mean()).max()) + 1e-12
            for j in range(2):
                ax[c + 1, j].set_ylim(-y, y)
            ax[c + 1, 1].tick_params(axis="y", labelleft=False)
        for j in range(2):
            ax[0, j].set_ylim(-0.3, min(max(cmax, TRIG_ON) * 1.1, PAPER_CFT_YMAX))
        ax[0, 0].set_ylabel("STA/LTA ratio")
        ax[0, 1].tick_params(axis="y", labelleft=False)
        ax[0, 1].legend(loc="upper right", ncol=2, fontsize=7.5)
        for j in range(2):
            ax[-1, j].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
            ax[-1, j].set_xlabel("UTC, %s" % str(tax[0])[:10])
        h = [Line2D([], [], ls="", marker="x", color=INK, ms=8, mew=1.6, label="STA/LTA trigger"),
             Line2D([], [], ls="", marker="o", ms=5, color=C_4, mec=INK, label="TauP P"),
             Line2D([], [], ls="", marker="o", ms=9, color=C_4, mec=INK, label="TauP S"),
             Line2D([], [], color=MUTED, lw=0.8, label="model window start (60 s windows, %d %% overlap)" % int(100 * OVERLAP))]
        fig.legend(handles=h, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.03))
        if case == "uncatalogued_trigger":
            what = "model trigger with no catalogue event within 2 min"
        elif case == "user_time":
            what = "requested time %s" % row.origin
        else:
            what = "M%.1f at %.0f km, origin %s" % (row.mag, row.dist_deg * 111.2, str(row.origin)[:19].replace("T", " "))
        fig.suptitle("%s, %s: %s  [%s]" % (tag, lab, what, case.replace("_", " ")), x=0.01, ha="left", fontsize=9.5)
        fig.tight_layout()
        save_fig(fig, str(Path("paper_stalta") / fname))

    def paper_daily(tag, trig, mid):
        S = STREAMS[tag]; ev = S["events"]
        nd = int(np.ceil(len(S["w"]) / FS / 86400))
        ids = [k for k in ("raw", "bandpass", mid) if k in trig]
        colour = {"raw": INK2, "bandpass": MUTED}
        wbar = 0.8 / len(ids)
        fig, ax = plt.subplots(figsize=(10.2, 3.6))
        for k, m in enumerate(ids):
            cnt = np.bincount((trig[m] // int(86400 * FS)).astype(int), minlength=nd)[:nd]
            ax.bar(np.arange(nd) + 0.1 + (k + 0.5) * wbar, cnt, wbar, color=colour.get(m, bcolor(meta(m)["kind"])), zorder=3,
                   label="%s: %d triggers" % ({"raw": "raw waveform", "bandpass": "band-pass %g-%g Hz" % BANDPASS_HZ}.get(m, "separated, " + m.replace("_seed99", "")), len(trig[m])))
        c2 = np.bincount((ev[ev.mag >= 2].tP.values // 86400).astype(int), minlength=nd)[:nd]
        ax.plot(np.arange(nd) + 0.5, c2, "-o", color=C_4, mec=INK, mew=0.8, ms=5, lw=1.4, zorder=5, label="M2+ local events in the catalogue")
        ax.set_yscale("log")
        ax.set_xlim(0, nd)
        ax.set_xlabel("days since %s" % START)
        ax.set_ylabel("per day")
        border(ax)
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=4, fontsize=7.5)
        ax.set_title("%s: STA/LTA %g/%g s, trigger on/off %g/%g" % (tag, STA_S, LTA_S, TRIG_ON, TRIG_OFF), loc="left")
        fig.tight_layout()
        save_fig(fig, str(Path("paper_stalta") / ("%s_%s_daily" % (tag, mid.replace("_seed99", "")))))

    for tag in STREAMS:
        STREAMS_START = STREAMS[tag]["meta"]["start"]
        raw, cf_raw, t_raw = run_detector(tag, "raw")
        t_bp = run_detector(tag, "bandpass")[2]
        for mid in PAPER_MODELS:
            if mid not in MODELS:
                print("  %s not among the loaded models, skipped" % mid); continue
            t1 = time.time()
            sep, cf_sep, t_sep = run_detector(tag, mid)
            paper_daily(tag, {"raw": t_raw, "bandpass": t_bp, mid: t_sep}, mid)
            picks = pick_examples(STREAMS[tag]["events"], t_raw, t_bp, t_sep)
            rows = []
            for n, (case, r) in enumerate(picks[:PAPER_N_EXAMPLES] + [p for p in picks if p[0] == "user_time"]):
                paper_zoom(tag, mid, raw, cf_raw, t_raw, sep, cf_sep, t_sep, float(r.tP), case, r,
                           "%s_%s_%d_%s" % (tag, mid.replace("_seed99", ""), n + 1, case))
                rows.append(dict(case=case, origin=r.origin, mag=r.mag, dist_km=r.dist_deg * 111.2, tP_s=float(r.tP),
                                 raw=bool(r.raw), bandpass=bool(r.bandpass), model=bool(r.model)))
            pd.DataFrame(rows).to_csv(RESULTS / ("paper_examples_%s_%s.csv" % (tag, mid)), index=False)
            del sep, cf_sep
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            log("Part 6: %s %s, %d example figures + daily (%.0f s)" % (tag, mid, len(rows), time.time() - t1))
else:
    print("Part 6 skipped")

  figure: paper_stalta/HV.HAT_stead_LSTM_daily (.pdf, .png)
  figure: paper_stalta/HV.HAT_stead_LSTM_1_detected_by_both (.pdf, .png)
  figure: paper_stalta/HV.HAT_stead_LSTM_2_separated_only (.pdf, .png)
  figure: paper_stalta/HV.HAT_stead_LSTM_3_separated_only (.pdf, .png)
  figure: paper_stalta/HV.HAT_stead_LSTM_4_missed_by_model (.pdf, .png)
  figure: paper_stalta/HV.HAT_stead_LSTM_5_uncatalogued_trigger (.pdf, .png)
- 2026-11-23 13:13:40  [08] Part 6: HV.HAT stead_LSTM_seed99, 5 example figures + daily (37 s)
  figure: paper_stalta/HV.HAT_pretrained_LSTM_daily (.pdf, .png)
  figure: paper_stalta/HV.HAT_pretrained_LSTM_1_detected_by_both (.pdf, .png)
  figure: paper_stalta/HV.HAT_pretrained_LSTM_2_separated_only (.pdf, .png)
  figure: paper_stalta/HV.HAT_pretrained_LSTM_3_separated_only (.pdf, .png)
  figure: paper_stalta/HV.HAT_pretrained_LSTM_4_missed_by_model (.pdf, .png)
  figure: paper_stalta/HV.HAT_pretrained_LSTM_5_uncatalogued_trigger (.pdf, .png)
- 2026-11-23 13:14:17  [08]